# 04 — Question Generation Demo

Full pipeline: **chunks (MongoDB) → select → prompt → Gemini LLM → parse → questions**

This notebook demonstrates Week 4:
1. Generate questions for a single Java chapter
2. Generate for the entire Java textbook
3. Generate for the C textbook
4. Save all questions + manual evaluation template

In [1]:
import json
import logging
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

from hsc_edu.storage.mongo_store import MongoChunkStore

mongo = MongoChunkStore()
print(f"MongoDB chunks: {mongo.count()}")
print(f"Subjects: {mongo.distinct_values('subject')}")
print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")

MongoDB chunks: 437
Subjects: ['Lập trình C', 'Lập trình Java']
GOOGLE_API_KEY set: True


## 1. Single chapter test — Java "7.7. ĐA HÌNH"

In [2]:
from hsc_edu.generation import generate_questions, GeneratedQuestion

qs_poly = generate_questions(
    subject="Lập trình Java",
    chapter="7.7. ĐA HÌNH",
    num_questions=3,
)

print(f"Generated {len(qs_poly)} questions:\n")
for i, q in enumerate(qs_poly, 1):
    print(f"[{i}] ({q.difficulty}) {q.question}")
    print(f"    Answer: {q.suggested_answer[:150]}..." if len(q.suggested_answer) > 150 else f"    Answer: {q.suggested_answer}")
    print(f"    Source: {q.source}  |  Bloom: {q.bloom_level}")
    print(f"    Keywords: {q.keywords}")
    print()

INFO | hsc_edu.generation.question_generator | Selected 3 chunks for generation (subject='Lập trình Java', chapter='7.7. ĐA HÌNH')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 2096 chars (model=gemini-flash-latest, attempt=1)
ERROR | hsc_edu.generation.question_generator | Failed to parse LLM output as JSON: [
  {
    "question": "Theo giáo trình, hai đặc điểm chính của tính đa hình trong lập trình hướng đối tượng là gì?",
    "suggested_answer": "Đa hình có hai đặc điểm: (1) Các đối tượng thuộc các lớp d…
INFO | hsc_edu.generation.question_generator | Generated 0 questions for subject='Lập trình Java' chapter='7.7. ĐA HÌNH'


Generated 0 questions:



## 2. Limited textbook (API-safe) — Java.pdf

In [3]:
import time

from hsc_edu.generation import generate_questions

JAVA_QUESTIONS_PER_CHAPTER = 1
JAVA_MAX_CHAPTERS = 5  # tăng để sinh nhiều hơn, giảm để tiết kiệm API

all_chunks_java = mongo.get_chunks_by_filter(subject="Lập trình Java")
chapters_java = sorted({c.chapter for c in all_chunks_java if c.chapter})
chapters_java = chapters_java[:JAVA_MAX_CHAPTERS]

qs_java = []
for i, ch in enumerate(chapters_java, 1):
    qs_java.extend(
        generate_questions(
            subject="Lập trình Java",
            chapter=ch,
            num_questions=JAVA_QUESTIONS_PER_CHAPTER,
            max_context_chunks=15,
            mongo=mongo,
        )
    )
    if i < len(chapters_java):
        time.sleep(1)

print(
    f"\nJava (limited): {len(qs_java)} questions from {len({q.chapter for q in qs_java})} chapters"
)
diff_dist = Counter(q.difficulty for q in qs_java)
print(f"Difficulty distribution: {dict(diff_dist)}")

INFO | hsc_edu.generation.question_generator | Selected 1 chunks for generation (subject='Lập trình Java', chapter='1.1. KHÁI NIỆM CƠ BẢN')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 782 chars (model=gemini-flash-latest, attempt=1)
INFO | hsc_edu.generation.question_generator | Generated 1 questions for subject='Lập trình Java' chapter='1.1. KHÁI NIỆM CƠ BẢN'
INFO | hsc_edu.generation.question_generator | Selected 3 chunks for generation (subject='Lập trình Java', chapter='1.2. ĐỐI TƯỢNG VÀ LỚP')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 2809 


Java (limited): 17 questions from 5 chapters
Difficulty distribution: {'Thông hiểu': 9, 'Nhận biết': 4, 'Vận dụng': 3, 'Vận dụng cao': 1}


## 3. Limited textbook (API-safe) — C.pdf

In [4]:
import time

from hsc_edu.generation import generate_questions

C_QUESTIONS_PER_CHAPTER = 1
C_MAX_CHAPTERS = 5

all_chunks_c = mongo.get_chunks_by_filter(subject="Lập trình C")
chapters_c = sorted({c.chapter for c in all_chunks_c if c.chapter})
chapters_c = chapters_c[:C_MAX_CHAPTERS]

qs_c = []
for i, ch in enumerate(chapters_c, 1):
    qs_c.extend(
        generate_questions(
            subject="Lập trình C",
            chapter=ch,
            num_questions=C_QUESTIONS_PER_CHAPTER,
            max_context_chunks=15,
            mongo=mongo,
        )
    )
    if i < len(chapters_c):
        time.sleep(1)

print(
    f"\nC (limited): {len(qs_c)} questions from {len({q.chapter for q in qs_c})} chapters"
)
diff_dist_c = Counter(q.difficulty for q in qs_c)
print(f"Difficulty distribution: {dict(diff_dist_c)}")

INFO | hsc_edu.generation.question_generator | Selected 1 chunks for generation (subject='Lập trình C', chapter='1.3.2. Sử dụng lưu đồ (Flowchart)')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_client | LLM generated 542 chars (model=gemini-flash-latest, attempt=1)
INFO | hsc_edu.generation.question_generator | Generated 1 questions for subject='Lập trình C' chapter='1.3.2. Sử dụng lưu đồ (Flowchart)'
INFO | hsc_edu.generation.question_generator | Selected 5 chunks for generation (subject='Lập trình C', chapter='2.1. Cài đặt và sử dụng Dev-C++')
INFO | google_genai.models | AFC is enabled with max remote calls: 10.
INFO | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-flash-latest:generateContent "HTTP/1.1 200 OK"
INFO | hsc_edu.generation.llm_cli


C (limited): 9 questions from 5 chapters
Difficulty distribution: {'Thông hiểu': 5, 'Vận dụng': 1, 'Nhận biết': 3}


## 4. Combine & save all questions

In [5]:
all_questions = qs_java + qs_c
print(f"Total questions: {len(all_questions)}")
print(f"  Java: {len(qs_java)}")
print(f"  C:    {len(qs_c)}")

output_path = Path("../data/questions/generated_questions.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        [q.model_dump() for q in all_questions],
        f, ensure_ascii=False, indent=2,
    )

print(f"\nSaved to {output_path}")

print("\n--- Sample questions (first 10) ---\n")
for i, q in enumerate(all_questions[:10], 1):
    print(f"[{i}] ({q.subject} | {q.chapter}) [{q.difficulty}]")
    print(f"    Q: {q.question}")
    print(f"    A: {q.suggested_answer[:120]}{'...' if len(q.suggested_answer) > 120 else ''}")
    print()

Total questions: 26
  Java: 17
  C:    9

Saved to ..\data\questions\generated_questions.json

--- Sample questions (first 10) ---

[1] (Lập trình Java | 1.1. KHÁI NIỆM CƠ BẢN) [Thông hiểu]
    Q: Dựa trên nội dung giáo trình, hãy giải thích bản chất của một chương trình hướng đối tượng khi đang vận hành (runtime) và cơ chế tương tác giữa các đối tượng thông qua ví dụ cụ thể.
    A: Trong thời gian chạy, chương trình OOP là một tập các đối tượng gửi thông điệp cho nhau để yêu cầu và thực hiện dịch vụ....

[2] (Lập trình Java | 1.2. ĐỐI TƯỢNG VÀ LỚP) [Thông hiểu]
    Q: Dựa trên giáo trình, hãy giải thích mối quan hệ giữa 'lớp' (class) và 'đối tượng' (object/instance). Tại sao nói chương trình khi chạy là một tập hợp các đối tượng tương tác?
    A: Lớp là khuôn mẫu/đặc tả các đặc điểm chung (thuộc tính và phương thức), còn đối tượng là một thực thể cụ thể được tạo ra...

[3] (Lập trình Java | 1.2. ĐỐI TƯỢNG VÀ LỚP) [Nhận biết]
    Q: Khi một đối tượng vừa được tạo ra, hệ thống tự động t

## 5. Manual evaluation — 20 questions

Select 20 random questions for manual evaluation.
Fill in the **Rating** column: `Đúng` / `Chưa tốt` / `Sai`

In [6]:
import random

random.seed(42)
eval_sample = random.sample(all_questions, min(20, len(all_questions)))

print("| # | Subject | Chapter | Difficulty | Question | Answer (preview) | Rating |")
print("|---|---------|---------|------------|----------|------------------|--------|")
for i, q in enumerate(eval_sample, 1):
    subj_short = q.subject.replace("Lập trình ", "")
    ch_short = q.chapter[:30] + "..." if len(q.chapter) > 30 else q.chapter
    q_short = q.question[:60] + "..." if len(q.question) > 60 else q.question
    a_short = q.suggested_answer[:50] + "..." if len(q.suggested_answer) > 50 else q.suggested_answer
    print(f"| {i} | {subj_short} | {ch_short} | {q.difficulty} | {q_short} | {a_short} | |")

| # | Subject | Chapter | Difficulty | Question | Answer (preview) | Rating |
|---|---------|---------|------------|----------|------------------|--------|
| 1 | C | 2.1. Cài đặt và sử dụng Dev-C+... | Nhận biết | Để có thể sử dụng các tính năng gỡ lỗi (Debug) trong Dev-C++... | Vào menu 'Tools' -> 'Compiler Options' -> tab 'Set... | |
| 2 | Java | 1.2. ĐỐI TƯỢNG VÀ LỚP | Thông hiểu | Trong lập trình hướng đối tượng, việc 'gửi thông điệp' giữa ... | Gửi thông điệp thực chất là gọi một phương thức củ... | |
| 3 | Java | 1.1. KHÁI NIỆM CƠ BẢN | Thông hiểu | Dựa trên nội dung giáo trình, hãy giải thích bản chất của mộ... | Trong thời gian chạy, chương trình OOP là một tập ... | |
| 4 | Java | 1.3. CÁC NGUYÊN TẮC TRỤ CỘT | Vận dụng | Giải thích mối quan hệ giữa 'Đóng gói' và 'Che giấu thông ti... | Đóng gói đi kèm với che giấu thông tin để ẩn đi cá... | |
| 5 | Java | 1.3. CÁC NGUYÊN TẮC TRỤ CỘT | Thông hiểu | Phân biệt khái niệm 'Thuộc tính' (attribute) và 'Trạng thái'... | Thuộc tính (bi

## 6. Statistics

In [7]:
print(f"=== Generation Statistics ===")
print(f"Total questions: {len(all_questions)}")
print(f"")

# By subject
by_subject = Counter(q.subject for q in all_questions)
print("By subject:")
for subj, cnt in by_subject.most_common():
    print(f"  {subj}: {cnt}")

# By difficulty
print("\nBy difficulty:")
by_diff = Counter(q.difficulty for q in all_questions)
for diff, cnt in by_diff.most_common():
    print(f"  {diff}: {cnt}")

# By bloom level
print("\nBy Bloom level:")
by_bloom = Counter(q.bloom_level for q in all_questions)
for bl, cnt in by_bloom.most_common():
    print(f"  {bl}: {cnt}")

# Chapter coverage
java_chapters_covered = len({q.chapter for q in qs_java if q.chapter})
c_chapters_covered = len({q.chapter for q in qs_c if q.chapter})
print(f"\nChapter coverage:")
print(f"  Java: {java_chapters_covered} chapters")
print(f"  C:    {c_chapters_covered} chapters")

=== Generation Statistics ===
Total questions: 26

By subject:
  Lập trình Java: 17
  Lập trình C: 9

By difficulty:
  Thông hiểu: 14
  Nhận biết: 7
  Vận dụng: 4
  Vận dụng cao: 1

By Bloom level:
  Understand: 14
  Remember: 7
  Apply: 4
  Analyze: 1

Chapter coverage:
  Java: 5 chapters
  C:    5 chapters
